In [ ]:
from google.colab import files

uploaded = files.upload()
print(uploaded.keys())

Saving final_regression_dataset_v1.xlsx to final_regression_dataset_v1.xlsx
dict_keys(['final_regression_dataset_v1.xlsx'])


In [ ]:
import pandas as pd
import numpy as np

file_name = list(uploaded.keys())[0]
file_path = f"/content/{file_name}"

reg = pd.read_excel(file_path, sheet_name="regression_base")
diag = pd.read_excel(file_path, sheet_name="diagnostics_desc")
macro = pd.read_excel(file_path, sheet_name="macro_zone_rates")

print(reg.shape)
reg.head()

(781, 29)


,firm_id,ticker,company_name,country,macro_zone,sector,fiscal_year,roa,roa_ebit,interest_coverage,...,revenue,ebit,ebitda,interest_expense,net_income,total_assets,total_debt,source_primary,data_quality_score,notes
0,F001,AAPL,Apple Inc.,United States,US,Technology,2022,0.282924,0.323701,40.749574,...,3.943280e+11,1.194370e+11,1.305410e+11,2.931000e+09,9.980300e+10,3.527550e+11,1.118240e+11,Yahoo + SEC override,1.000,NaN
1,F001,AAPL,Apple Inc.,United States,US,Technology,2023,0.275098,0.323701,29.062039,...,3.832850e+11,1.143010e+11,1.258200e+11,3.933000e+09,9.699500e+10,3.525830e+11,1.065720e+11,Yahoo + SEC override,1.000,NaN
2,F001,AAPL,Apple Inc.,United States,US,Technology,2024,0.256825,0.323701,NaN,...,3.910350e+11,1.232160e+11,1.346610e+11,NaN,9.373600e+10,3.649800e+11,9.734100e+10,Yahoo + SEC override,0.875,NaN
3,F001,AAPL,Apple Inc.,United States,US,Technology,2025,0.306894,0.323701,NaN,...,4.161610e+11,1.330500e+11,1.447480e+11,NaN,1.120100e+11,3.649800e+11,9.128100e+10,Yahoo + SEC override,0.875,NaN
4,F002,MSFT,Microsoft Corporation,United States,US,Technology,2022,0.121371,0.145157,20.439599,...,1.430150e+11,5.295900e+10,1.002390e+11,2.591000e+09,4.428100e+10,3.648400e+11,5.551100e+10,Yahoo + SEC override,1.000,NaN


In [ ]:
s = reg["interest_coverage"].dropna()

print("Min:", s.min())
print("P1:", s.quantile(0.01))
print("Median:", s.median())
print("P99:", s.quantile(0.99))
print("Max:", s.max())

Min: -597.3913043478261
P1: -3.974792574809226
Median: 11.080523274274466
P99: 806.6476085434132
Max: 16438.81481481481


In [ ]:
# on garde une copie brute
reg["interest_coverage_raw_backup"] = reg["interest_coverage"]

# seuils
lo = reg["interest_coverage"].quantile(0.01)
hi = reg["interest_coverage"].quantile(0.99)

# winsorisation
reg["interest_coverage"] = reg["interest_coverage"].clip(lower=lo, upper=hi)

print("Lower bound:", lo)
print("Upper bound:", hi)

changed = (
    (reg["interest_coverage_raw_backup"] < lo) |
    (reg["interest_coverage_raw_backup"] > hi)
).sum()

print("Nombre de lignes modifiées :", changed)

Lower bound: -3.974792574809226
Upper bound: 806.6476085434132
Nombre de lignes modifiées : 16


In [ ]:
s2 = reg["interest_coverage"].dropna()

print("Min après correction:", s2.min())
print("P1 après correction:", s2.quantile(0.01))
print("Median après correction:", s2.median())
print("P99 après correction:", s2.quantile(0.99))
print("Max après correction:", s2.max())

Min après correction: -3.974792574809226
P1 après correction: -3.9414912429049136
Median après correction: 11.080523274274466
P99 après correction: 761.774979083904
Max après correction: 806.6476085434132


In [ ]:
num_cols = [
    "roa",
    "roa_ebit",
    "interest_coverage",
    "debt_ebitda",
    "ebitda_margin",
    "book_leverage",
    "policy_rate_fye",
    "policy_rate_change",
    "leverage_lag",
    "rate_x_leverage_lag",
    "rate_change_x_leverage_lag",
    "size_ln_assets"
]

diagnostics_desc_new = reg[num_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

diagnostics_desc_new["missing_pct"] = reg[num_cols].isna().mean().values

diagnostics_desc_new.head()

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_pct
roa,781.0,0.083080,0.070410,-0.321651,-0.075651,0.002907,0.041999,0.070569,0.117689,0.203085,0.276664,0.653041,0.000000
roa_ebit,781.0,0.112353,0.071872,-0.057531,-0.057151,0.018704,0.065270,0.098915,0.150459,0.251328,0.323605,0.323701,0.000000
interest_coverage,774.0,37.229878,105.797231,-3.974793,-3.941491,1.665390,6.099205,11.080523,24.203014,107.224085,761.774979,806.647609,0.008963
debt_ebitda,742.0,2.208239,1.991132,0.011371,0.033227,0.232343,0.973946,1.850978,2.870219,4.620355,13.555798,13.625877,0.049936
ebitda_margin,774.0,0.294960,0.211198,0.019581,0.020176,0.071988,0.160796,0.254144,0.364961,0.604591,1.336838,1.353929,0.008963


In [ ]:
output_path = "/content/final_regression_dataset_v2.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    reg.to_excel(writer, sheet_name="regression_base", index=False)
    diagnostics_desc_new.to_excel(writer, sheet_name="diagnostics_desc")
    macro.to_excel(writer, sheet_name="macro_zone_rates", index=False)

print("Saved:", output_path)

Saved: /content/final_regression_dataset_v2.xlsx


In [ ]:
from google.colab import files
files.download("/content/final_regression_dataset_v2.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

file_path = "/content/final_regression_dataset_v2.xlsx"   # adapte si besoin
df = pd.read_excel(file_path, sheet_name="regression_base")

print(df.shape)
df.head()

(781, 30)


,firm_id,ticker,company_name,country,macro_zone,sector,fiscal_year,roa,roa_ebit,interest_coverage,...,ebit,ebitda,interest_expense,net_income,total_assets,total_debt,source_primary,data_quality_score,notes,interest_coverage_raw_backup
0,F001,AAPL,Apple Inc.,United States,US,Technology,2022,0.282924,0.323701,40.749574,...,1.194370e+11,1.305410e+11,2.931000e+09,9.980300e+10,3.527550e+11,1.118240e+11,Yahoo + SEC override,1.000,NaN,40.749574
1,F001,AAPL,Apple Inc.,United States,US,Technology,2023,0.275098,0.323701,29.062039,...,1.143010e+11,1.258200e+11,3.933000e+09,9.699500e+10,3.525830e+11,1.065720e+11,Yahoo + SEC override,1.000,NaN,29.062039
2,F001,AAPL,Apple Inc.,United States,US,Technology,2024,0.256825,0.323701,NaN,...,1.232160e+11,1.346610e+11,NaN,9.373600e+10,3.649800e+11,9.734100e+10,Yahoo + SEC override,0.875,NaN,NaN
3,F001,AAPL,Apple Inc.,United States,US,Technology,2025,0.306894,0.323701,NaN,...,1.330500e+11,1.447480e+11,NaN,1.120100e+11,3.649800e+11,9.128100e+10,Yahoo + SEC override,0.875,NaN,NaN
4,F002,MSFT,Microsoft Corporation,United States,US,Technology,2022,0.121371,0.145157,20.439599,...,5.295900e+10,1.002390e+11,2.591000e+09,4.428100e+10,3.648400e+11,5.551100e+10,Yahoo + SEC override,1.000,NaN,20.439599


In [ ]:
reg_cols = [
    "firm_id",
    "fiscal_year",
    "country",
    "macro_zone",
    "roa",
    "policy_rate_change",
    "leverage_lag",
    "rate_change_x_leverage_lag",
    "size_ln_assets"
]

reg1 = df[reg_cols].dropna().copy()

print("Observations:", len(reg1))
print("Firmes:", reg1["firm_id"].nunique())
print("Années:", sorted(reg1["fiscal_year"].unique()))
reg1.head()

Observations: 561
Firmes: 191
Années: [np.int64(2023), np.int64(2024), np.int64(2025)]


,firm_id,fiscal_year,country,macro_zone,roa,policy_rate_change,leverage_lag,rate_change_x_leverage_lag,size_ln_assets
1,F001,2023,United States,US,0.275098,1.23,0.317002,0.389912,26.588552
2,F001,2024,United States,US,0.256825,-0.85,0.302261,-0.256922,26.623108
3,F001,2025,United States,US,0.306894,-0.76,0.266702,-0.202694,26.623108
5,F002,2023,United States,US,0.198336,1.23,0.152152,0.187147,26.622725
6,F002,2024,United States,US,0.172086,-0.85,0.152152,-0.129329,26.961909


In [ ]:
print(reg1.describe())

print("\nRépartition par année")
print(reg1["fiscal_year"].value_counts().sort_index())

print("\nRépartition par macro_zone")
print(reg1["macro_zone"].value_counts())

       fiscal_year         roa  policy_rate_change  leverage_lag  \
count   561.000000  561.000000          561.000000    561.000000   
mean   2023.987522    0.081260            0.009418      0.275423   
std       0.813845    0.067694            1.144878      0.154306   
min    2023.000000   -0.121020           -1.546959      0.000000   
25%    2023.000000    0.041521           -0.850000      0.168762   
50%    2024.000000    0.069138           -0.760000      0.263625   
75%    2025.000000    0.112198            1.230000      0.357108   
max    2025.000000    0.653041            2.399477      0.845053   

       rate_change_x_leverage_lag  size_ln_assets  
count                  561.000000      561.000000  
mean                    -0.002192       25.185507  
std                      0.350404        1.710027  
min                     -0.777931       21.666666  
25%                     -0.235905       24.148473  
50%                     -0.084979       24.898852  
75%                    

In [ ]:
model_pool = smf.ols(
    "roa ~ policy_rate_change + leverage_lag + rate_change_x_leverage_lag + size_ln_assets",
    data=reg1
).fit(cov_type="cluster", cov_kwds={"groups": reg1["firm_id"]})

print(model_pool.summary())

                            OLS Regression Results                            
Dep. Variable:                    roa   R-squared:                       0.027
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     3.942
Date:                Tue, 31 Mar 2026   Prob (F-statistic):            0.00425
Time:                        13:28:41   Log-Likelihood:                 722.76
No. Observations:                 561   AIC:                            -1436.
Df Residuals:                     556   BIC:                            -1414.
Df Model:                           4                                         
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

In [ ]:
model_fe = smf.ols(
    "roa ~ policy_rate_change + leverage_lag + rate_change_x_leverage_lag + size_ln_assets + C(firm_id) + C(fiscal_year)",
    data=reg1
).fit(cov_type="cluster", cov_kwds={"groups": reg1["firm_id"]})

print(model_fe.summary())

                            OLS Regression Results                            
Dep. Variable:                    roa   R-squared:                       0.800
Model:                            OLS   Adj. R-squared:                  0.692
Method:                 Least Squares   F-statistic:                     1856.
Date:                Tue, 31 Mar 2026   Prob (F-statistic):          9.76e-166
Time:                        13:28:52   Log-Likelihood:                 1166.3
No. Observations:                 561   AIC:                            -1939.
Df Residuals:                     364   BIC:                            -1086.
Df Model:                         196                                         
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 196, but rank is 6
  warnings.warn('covariance of constraints does not have full '


In [ ]:
main_vars = [
    "policy_rate_change",
    "leverage_lag",
    "rate_change_x_leverage_lag",
    "size_ln_assets"
]

results_main = pd.DataFrame({
    "coef": model_fe.params[main_vars],
    "std_err": model_fe.bse[main_vars],
    "t_stat": model_fe.tvalues[main_vars],
    "p_value": model_fe.pvalues[main_vars]
})

print(results_main)

                                coef   std_err    t_stat   p_value
policy_rate_change          0.000008  0.004009  0.002029  0.998381
leverage_lag               -0.017183  0.096941 -0.177253  0.859309
rate_change_x_leverage_lag  0.010151  0.013774  0.736995  0.461125
size_ln_assets              0.039839  0.130286  0.305779  0.759773


In [ ]:
print("N observations :", int(model_fe.nobs))
print("R² :", round(model_fe.rsquared, 4))
print("Adj. R² :", round(model_fe.rsquared_adj, 4))

print("\nCoefficient clé :")
print("rate_change_x_leverage_lag =", round(model_fe.params["rate_change_x_leverage_lag"], 6))
print("p-value =", round(model_fe.pvalues["rate_change_x_leverage_lag"], 6))

N observations : 561
R² : 0.7998
Adj. R² : 0.692

Coefficient clé :
rate_change_x_leverage_lag = 0.010151
p-value = 0.461125


In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

file_path = "/content/final_regression_dataset_v2.xlsx"   # adapte si besoin
df = pd.read_excel(file_path, sheet_name="regression_base")

reg_cols = [
    "firm_id",
    "fiscal_year",
    "roa_ebit",
    "policy_rate_change",
    "leverage_lag",
    "rate_change_x_leverage_lag",
    "size_ln_assets"
]

reg2 = df[reg_cols].dropna().copy()

print("Observations:", len(reg2))
print("Firmes:", reg2["firm_id"].nunique())
print("Années:", sorted(reg2["fiscal_year"].unique()))

model_roa_ebit = smf.ols(
    "roa_ebit ~ policy_rate_change + leverage_lag + rate_change_x_leverage_lag + size_ln_assets + C(firm_id) + C(fiscal_year)",
    data=reg2
).fit(cov_type="cluster", cov_kwds={"groups": reg2["firm_id"]})

print(model_roa_ebit.summary())

Observations: 561
Firmes: 191
Années: [np.int64(2023), np.int64(2024), np.int64(2025)]


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 196, but rank is 6
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:               roa_ebit   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.806
Method:                 Least Squares   F-statistic:                     2179.
Date:                Tue, 31 Mar 2026   Prob (F-statistic):          3.11e-172
Time:                        13:41:53   Log-Likelihood:                 1278.1
No. Observations:                 561   AIC:                            -2162.
Df Residuals:                     364   BIC:                            -1309.
Df Model:                         196                                         
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

In [ ]:
main_vars = [
    "policy_rate_change",
    "leverage_lag",
    "rate_change_x_leverage_lag",
    "size_ln_assets"
]

results_roa_ebit = pd.DataFrame({
    "coef": model_roa_ebit.params[main_vars],
    "std_err": model_roa_ebit.bse[main_vars],
    "t_stat": model_roa_ebit.tvalues[main_vars],
    "p_value": model_roa_ebit.pvalues[main_vars]
})

print(results_roa_ebit)

print("\nN observations :", int(model_roa_ebit.nobs))
print("R² :", round(model_roa_ebit.rsquared, 4))
print("Adj. R² :", round(model_roa_ebit.rsquared_adj, 4))

print("\nCoefficient clé :")
print("rate_change_x_leverage_lag =", round(model_roa_ebit.params["rate_change_x_leverage_lag"], 6))
print("p-value =", round(model_roa_ebit.pvalues["rate_change_x_leverage_lag"], 6))

                                coef   std_err    t_stat   p_value
policy_rate_change         -0.002777  0.003752 -0.740109  0.459234
leverage_lag                0.036917  0.080540  0.458364  0.646691
rate_change_x_leverage_lag  0.016263  0.012479  1.303222  0.192499
size_ln_assets              0.017288  0.055047  0.314060  0.753476

N observations : 561
R² : 0.8737
Adj. R² : 0.8056

Coefficient clé :
rate_change_x_leverage_lag = 0.016263
p-value = 0.192499


In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

file_path = "/content/final_regression_dataset_v2.xlsx"   # adapte si besoin
df = pd.read_excel(file_path, sheet_name="regression_base")

reg_cols = [
    "firm_id",
    "fiscal_year",
    "interest_coverage",
    "policy_rate_change",
    "leverage_lag",
    "rate_change_x_leverage_lag",
    "size_ln_assets"
]

reg3 = df[reg_cols].dropna().copy()

print("Observations:", len(reg3))
print("Firmes:", reg3["firm_id"].nunique())
print("Années:", sorted(reg3["fiscal_year"].unique()))

model_ic = smf.ols(
    "interest_coverage ~ policy_rate_change + leverage_lag + rate_change_x_leverage_lag + size_ln_assets + C(firm_id) + C(fiscal_year)",
    data=reg3
).fit(cov_type="cluster", cov_kwds={"groups": reg3["firm_id"]})

print(model_ic.summary())

Observations: 556
Firmes: 190
Années: [np.int64(2023), np.int64(2024), np.int64(2025)]
                            OLS Regression Results                            
Dep. Variable:      interest_coverage   R-squared:                       0.636
Model:                            OLS   Adj. R-squared:                  0.440
Method:                 Least Squares   F-statistic:                     23.07
Date:                Tue, 31 Mar 2026   Prob (F-statistic):           2.37e-20
Time:                        13:46:42   Log-Likelihood:                -2912.5
No. Observations:                 556   AIC:                             6217.
Df Residuals:                     360   BIC:                             7064.
Df Model:                         195                                         
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 195, but rank is 6
  warnings.warn('covariance of constraints does not have full '


In [ ]:
main_vars = [
    "policy_rate_change",
    "leverage_lag",
    "rate_change_x_leverage_lag",
    "size_ln_assets"
]

results_ic = pd.DataFrame({
    "coef": model_ic.params[main_vars],
    "std_err": model_ic.bse[main_vars],
    "t_stat": model_ic.tvalues[main_vars],
    "p_value": model_ic.pvalues[main_vars]
})

print(results_ic)

print("\nN observations :", int(model_ic.nobs))
print("R² :", round(model_ic.rsquared, 4))
print("Adj. R² :", round(model_ic.rsquared_adj, 4))

print("\nCoefficient clé :")
print("rate_change_x_leverage_lag =", round(model_ic.params["rate_change_x_leverage_lag"], 6))
print("p-value =", round(model_ic.pvalues["rate_change_x_leverage_lag"], 6))

                                  coef     std_err    t_stat   p_value
policy_rate_change          -13.990963   12.532394 -1.116384  0.264258
leverage_lag               -174.248008  169.405548 -1.028585  0.303675
rate_change_x_leverage_lag   11.509904   23.760149  0.484421  0.628087
size_ln_assets               80.708062   63.415400  1.272689  0.203129

N observations : 556
R² : 0.6365
Adj. R² : 0.4395

Coefficient clé :
rate_change_x_leverage_lag = 11.509904
p-value = 0.628087


In [2]:
import pandas as pd
import statsmodels.formula.api as smf

file_path = "/content/final_regression_dataset_v2.xlsx"   # adapte si besoin
df = pd.read_excel(file_path, sheet_name="regression_base")

print(df.shape)
df.head()

(781, 30)


,firm_id,ticker,company_name,country,macro_zone,sector,fiscal_year,roa,roa_ebit,interest_coverage,...,ebit,ebitda,interest_expense,net_income,total_assets,total_debt,source_primary,data_quality_score,notes,interest_coverage_raw_backup
0,F001,AAPL,Apple Inc.,United States,US,Technology,2022,0.282924,0.323701,40.749574,...,1.194370e+11,1.305410e+11,2.931000e+09,9.980300e+10,3.527550e+11,1.118240e+11,Yahoo + SEC override,1.000,NaN,40.749574
1,F001,AAPL,Apple Inc.,United States,US,Technology,2023,0.275098,0.323701,29.062039,...,1.143010e+11,1.258200e+11,3.933000e+09,9.699500e+10,3.525830e+11,1.065720e+11,Yahoo + SEC override,1.000,NaN,29.062039
2,F001,AAPL,Apple Inc.,United States,US,Technology,2024,0.256825,0.323701,NaN,...,1.232160e+11,1.346610e+11,NaN,9.373600e+10,3.649800e+11,9.734100e+10,Yahoo + SEC override,0.875,NaN,NaN
3,F001,AAPL,Apple Inc.,United States,US,Technology,2025,0.306894,0.323701,NaN,...,1.330500e+11,1.447480e+11,NaN,1.120100e+11,3.649800e+11,9.128100e+10,Yahoo + SEC override,0.875,NaN,NaN
4,F002,MSFT,Microsoft Corporation,United States,US,Technology,2022,0.121371,0.145157,20.439599,...,5.295900e+10,1.002390e+11,2.591000e+09,4.428100e+10,3.648400e+11,5.551100e+10,Yahoo + SEC override,1.000,NaN,20.439599


In [4]:
reg_cols = [
    "firm_id",
    "fiscal_year",
    "roa",
    "policy_rate_fye",
    "leverage_lag",
    "rate_x_leverage_lag",
    "size_ln_assets"
]

reg_level_roa = df[reg_cols].dropna().copy()

print("Observations:", len(reg_level_roa))
print("Firmes:", reg_level_roa["firm_id"].nunique())
print("Années:", sorted(reg_level_roa["fiscal_year"].unique()))

Observations: 561
Firmes: 191
Années: [np.int64(2023), np.int64(2024), np.int64(2025)]


In [5]:
model_level_roa = smf.ols(
    "roa ~ policy_rate_fye + leverage_lag + rate_x_leverage_lag + size_ln_assets + C(firm_id) + C(fiscal_year)",
    data=reg_level_roa
).fit(cov_type="cluster", cov_kwds={"groups": reg_level_roa["firm_id"]})

print(model_level_roa.summary())

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 196, but rank is 6
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:                    roa   R-squared:                       0.800
Model:                            OLS   Adj. R-squared:                  0.693
Method:                 Least Squares   F-statistic:                     376.6
Date:                Sun, 05 Apr 2026   Prob (F-statistic):          1.29e-102
Time:                        22:44:16   Log-Likelihood:                 1167.1
No. Observations:                 561   AIC:                            -1940.
Df Residuals:                     364   BIC:                            -1087.
Df Model:                         196                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -0

In [6]:
main_vars = [
    "policy_rate_fye",
    "leverage_lag",
    "rate_x_leverage_lag",
    "size_ln_assets"
]

results_level_roa = pd.DataFrame({
    "coef": model_level_roa.params[main_vars],
    "std_err": model_level_roa.bse[main_vars],
    "t_stat": model_level_roa.tvalues[main_vars],
    "p_value": model_level_roa.pvalues[main_vars]
})

print(results_level_roa)

print("\nN observations :", int(model_level_roa.nobs))
print("R² :", round(model_level_roa.rsquared, 4))
print("Adj. R² :", round(model_level_roa.rsquared_adj, 4))

print("\nCoefficient clé :")
print("rate_x_leverage_lag =", round(model_level_roa.params["rate_x_leverage_lag"], 6))
print("p-value =", round(model_level_roa.pvalues["rate_x_leverage_lag"], 6))

                         coef   std_err    t_stat   p_value
policy_rate_fye     -0.003687  0.006275 -0.587609  0.556795
leverage_lag        -0.103558  0.118897 -0.870986  0.383762
rate_x_leverage_lag  0.022505  0.018084  1.244479  0.213323
size_ln_assets       0.042686  0.124661  0.342415  0.732039

N observations : 561
R² : 0.8004
Adj. R² : 0.6929

Coefficient clé :
rate_x_leverage_lag = 0.022505
p-value = 0.213323


In [7]:
import pandas as pd
import statsmodels.formula.api as smf

file_path = "/content/final_regression_dataset_v2.xlsx"   # adapte si besoin
df = pd.read_excel(file_path, sheet_name="regression_base")

print(df.shape)
df.head()

(781, 30)


,firm_id,ticker,company_name,country,macro_zone,sector,fiscal_year,roa,roa_ebit,interest_coverage,...,ebit,ebitda,interest_expense,net_income,total_assets,total_debt,source_primary,data_quality_score,notes,interest_coverage_raw_backup
0,F001,AAPL,Apple Inc.,United States,US,Technology,2022,0.282924,0.323701,40.749574,...,1.194370e+11,1.305410e+11,2.931000e+09,9.980300e+10,3.527550e+11,1.118240e+11,Yahoo + SEC override,1.000,NaN,40.749574
1,F001,AAPL,Apple Inc.,United States,US,Technology,2023,0.275098,0.323701,29.062039,...,1.143010e+11,1.258200e+11,3.933000e+09,9.699500e+10,3.525830e+11,1.065720e+11,Yahoo + SEC override,1.000,NaN,29.062039
2,F001,AAPL,Apple Inc.,United States,US,Technology,2024,0.256825,0.323701,NaN,...,1.232160e+11,1.346610e+11,NaN,9.373600e+10,3.649800e+11,9.734100e+10,Yahoo + SEC override,0.875,NaN,NaN
3,F001,AAPL,Apple Inc.,United States,US,Technology,2025,0.306894,0.323701,NaN,...,1.330500e+11,1.447480e+11,NaN,1.120100e+11,3.649800e+11,9.128100e+10,Yahoo + SEC override,0.875,NaN,NaN
4,F002,MSFT,Microsoft Corporation,United States,US,Technology,2022,0.121371,0.145157,20.439599,...,5.295900e+10,1.002390e+11,2.591000e+09,4.428100e+10,3.648400e+11,5.551100e+10,Yahoo + SEC override,1.000,NaN,20.439599


In [8]:
reg_cols = [
    "firm_id",
    "fiscal_year",
    "book_leverage",
    "policy_rate_fye",
    "leverage_lag",
    "rate_x_leverage_lag",
    "size_ln_assets"
]

reg_level_lev = df[reg_cols].dropna().copy()

print("Observations:", len(reg_level_lev))
print("Firmes:", reg_level_lev["firm_id"].nunique())
print("Années:", sorted(reg_level_lev["fiscal_year"].unique()))

Observations: 560
Firmes: 190
Années: [np.int64(2023), np.int64(2024), np.int64(2025)]


In [9]:
model_level_lev = smf.ols(
    "book_leverage ~ policy_rate_fye + leverage_lag + rate_x_leverage_lag + size_ln_assets + C(firm_id) + C(fiscal_year)",
    data=reg_level_lev
).fit(cov_type="cluster", cov_kwds={"groups": reg_level_lev["firm_id"]})

print(model_level_lev.summary())

                            OLS Regression Results                            
Dep. Variable:          book_leverage   R-squared:                       0.960
Model:                            OLS   Adj. R-squared:                  0.939
Method:                 Least Squares   F-statistic:                     18.58
Date:                Sun, 05 Apr 2026   Prob (F-statistic):           6.14e-17
Time:                        22:51:52   Log-Likelihood:                 1149.3
No. Observations:                 560   AIC:                            -1907.
Df Residuals:                     364   BIC:                            -1058.
Df Model:                         195                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  2

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 195, but rank is 6
  warnings.warn('covariance of constraints does not have full '


In [10]:
main_vars = [
    "policy_rate_fye",
    "leverage_lag",
    "rate_x_leverage_lag",
    "size_ln_assets"
]

results_level_lev = pd.DataFrame({
    "coef": model_level_lev.params[main_vars],
    "std_err": model_level_lev.bse[main_vars],
    "t_stat": model_level_lev.tvalues[main_vars],
    "p_value": model_level_lev.pvalues[main_vars]
})

print(results_level_lev)

print("\nN observations :", int(model_level_lev.nobs))
print("R² :", round(model_level_lev.rsquared, 4))
print("Adj. R² :", round(model_level_lev.rsquared_adj, 4))

print("\nCoefficient clé :")
print("rate_x_leverage_lag =", round(model_level_lev.params["rate_x_leverage_lag"], 6))
print("p-value =", round(model_level_lev.pvalues["rate_x_leverage_lag"], 6))

                         coef   std_err    t_stat   p_value
policy_rate_fye     -0.005598  0.008010 -0.698882  0.484626
leverage_lag        -0.043128  0.242826 -0.177610  0.859029
rate_x_leverage_lag  0.011685  0.019912  0.586831  0.557317
size_ln_assets      -0.072617  0.045162 -1.607919  0.107853

N observations : 560
R² : 0.9601
Adj. R² : 0.9388

Coefficient clé :
rate_x_leverage_lag = 0.011685
p-value = 0.557317
